# 12 · Counting tokens and cost per turn
Level L2 · Part 2 · core · ~30 min · builds on 03, 08

## Goal
Turn every reply's `usage` into dollars with a price table, count a request's tokens before you send it,
and show with Scout's own 10-turn chat why a long chat's total cost grows roughly with the square of its length.

## The idea
Providers bill per token, with separate prices for **input** (everything you send) and **output** (everything
the model writes, including thinking). Prices are quoted **per million tokens**. Every response reports its
**usage**, so the cost of a turn is two multiplications. You can also ask the provider to **count tokens**
before sending, to check a request fits your budget. Lesson 08 showed that a chat resends its whole history
each turn. So turn 10 pays again for turns 1 to 9, and the total over a chat grows much faster than the chat.

Analogy: a taxi that charges per kilometre, where every new stop means driving back to the start and redoing
the whole route.

## Picture
```mermaid
flowchart LR
    H["history + new question"] --> C["count_tokens<br/>(before, free)"]
    C --> B{"within budget?"}
    B -->|yes| G["generate_content"]
    G --> U["usage: input, output, thinking"]
    U --> P["x price per million"]
    P --> T["$ this turn"] --> S["$ so far"]
```

## Step by step
### 1. Start the recorder
Model calls are recorded once to `../data/llm_cassettes/12_tokens-cost-per-turn.yaml` and replayed after
that, with no key (lesson 07).

In [1]:
import sys; sys.path.insert(0, "..")
import llm_replay
llm_replay.start("12_tokens-cost-per-turn")

LLM mode: replay (12_tokens-cost-per-turn.yaml)


### 2. Setup from lessons 07 and 08
Scout's notes, the pinned model id, the client, and Scout's system instruction. Nothing new.

In [2]:
import os
from dotenv import load_dotenv
from google import genai
from google.genai import types

NOTES = [
    {"id": 1, "text": "Mars has two small moons, Phobos and Deimos."},
    {"id": 2, "text": "Venus spins backwards compared with most planets."},
    {"id": 3, "text": "A day on Mars lasts about 24 hours and 39 minutes."},
    {"id": 4, "text": "Jupiter is the largest planet in the solar system."},
    {"id": 5, "text": "The rover Perseverance landed on Mars in February 2021."},
    {"id": 6, "text": "Saturn's rings are mostly made of ice."},
]
load_dotenv("../.env")
MODEL = os.environ.get("TUT_LLM_MODEL", "gemini-3.5-flash-lite")
client = genai.Client(vertexai=True, api_key=os.environ.get("GOOGLE_CLOUD_API_KEY") or "replay-no-key-needed")
NO_AFC = types.AutomaticFunctionCallingConfig(disable=True)
NOTES_TEXT = "\n".join(f"Note {n['id']}: {n['text']}" for n in NOTES)
SYSTEM_TEXT = ("You are Scout, a research assistant. Answer only from the user's notes below and this chat. "
               "If the notes do not cover it, say so in one sentence. Keep answers to one sentence.\n\n" + NOTES_TEXT)
print(len(NOTES), "notes; model:", MODEL)

6 notes; model: gemini-3.5-flash-lite


Lesson 08's message helpers, unchanged: Scout keeps a vendor-neutral `{"role", "text"}` list, checks it, and
converts it to Gemini's `types.Content` at the edge.

In [3]:
GEMINI_ROLES = {"user": "user", "assistant": "model"}

def turn(role: str, text: str) -> types.Content:
    return types.Content(role=role, parts=[types.Part(text=text)])

def to_gemini(messages: list[dict]) -> tuple[str | None, list[types.Content]]:
    system = messages[0]["text"] if messages and messages[0]["role"] == "system" else None
    rest = messages[1:] if system is not None else messages
    return system, [turn(GEMINI_ROLES[m["role"]], m["text"]) for m in rest]

def check_messages(messages: list[dict]) -> None:
    if not messages or messages[-1]["role"] != "user":
        raise ValueError("history must be non-empty and end with a user message")
    for i, m in enumerate(messages):
        if m["role"] not in ("system", "user", "assistant") or (m["role"] == "system" and i):
            raise ValueError(f"message {i}: bad role {m['role']!r}")

print("helpers ready")

helpers ready


### 3. A price table
Prices are per **million** tokens, input and output separately. These are the list prices for the course's
model from RESEARCH.md (checked 2026-10-06). Prices change and differ by platform, so they live in one table
with a date, never in the code that uses them.

In [4]:
PRICES_PER_MTOK = {            # USD per 1,000,000 tokens; source: Gemini pricing page, 2026-10-06
    "gemini-3.5-flash-lite": {"input": 0.30, "output": 2.50},
    "gemini-3.8-flash":      {"input": 0.75, "output": 3.75},   # promo price until 2026-12-31, then 1.50 / 7.50
}

def cost_usd(model: str, input_tokens: int, output_tokens: int) -> float:
    price = PRICES_PER_MTOK[model]
    return (input_tokens * price["input"] + output_tokens * price["output"]) / 1_000_000

print(f"1,000 in + 200 out on {MODEL}: ${cost_usd(MODEL, 1_000, 200):.5f}")

1,000 in + 200 out on gemini-3.5-flash-lite: $0.00080


The smallest example, no model needed: 1,000 input tokens cost $0.0003 and 200 output tokens cost $0.0005.
Output is more than 8 times dearer per token here, so a chatty reply can cost more than a long prompt.

### 4. Cost of one real call
Each response carries `usage_metadata` (lesson 07). Input is `prompt_token_count`. Output is
`candidates_token_count` plus `thoughts_token_count`, because thinking is billed as output. Either can be
`None`, so we default to 0.

In [5]:
def usage_tokens(resp) -> tuple[int, int]:
    u = resp.usage_metadata
    return u.prompt_token_count or 0, (u.candidates_token_count or 0) + (u.thoughts_token_count or 0)

def scout_config(max_output_tokens: int = 60) -> types.GenerateContentConfig:
    return types.GenerateContentConfig(system_instruction=SYSTEM_TEXT, max_output_tokens=max_output_tokens,
                                       automatic_function_calling=NO_AFC)

print("ready")

ready


One question to Scout, then its bill.

In [6]:
resp = client.models.generate_content(model=MODEL, contents="When did Perseverance land?", config=scout_config())
tokens_in, tokens_out = usage_tokens(resp)
print("reply:", resp.text.strip())
print(f"usage: {tokens_in} in, {tokens_out} out -> ${cost_usd(MODEL, tokens_in, tokens_out):.6f}")

reply: The rover Perseverance landed on Mars in February 2021.
usage: 146 in, 14 out -> $0.000079


If you skipped the model cell (no key, no recording), this sets the recorded usage of that call so the
rest of the lesson still runs.

In [7]:
if "resp" not in globals():   # recorded output of the call above: 146 in, 14 out, no thinking
    resp = types.GenerateContentResponse(usage_metadata=types.GenerateContentResponseUsageMetadata(
        prompt_token_count=146, candidates_token_count=14))
    tokens_in, tokens_out = usage_tokens(resp)
print(f"tokens_in={tokens_in}, tokens_out={tokens_out}")

tokens_in=146, tokens_out=14


Most of the input is the system instruction with the six notes, not the five-word question. Scout pays
for its rules on every single call.

### 5. Count tokens before sending
`client.models.count_tokens` asks Gemini's own tokenizer how many input tokens a request would use, without
generating anything. Google documents it as free of charge, with its own quota (3,000 requests per minute).
Pass the system instruction too, or you undercount.

In [8]:
question = "When did Perseverance land?"
bare = client.models.count_tokens(model=MODEL, contents=question)
full = client.models.count_tokens(model=MODEL, contents=question,
                                  config=types.CountTokensConfig(system_instruction=SYSTEM_TEXT))
print("question only:", bare.total_tokens, "| with system instruction:", full.total_tokens,
      "| usage after the call:", tokens_in)

question only: 5 | with system instruction: 146 | usage after the call: 146


The full count matches the `prompt_token_count` the real call reported. Counting the question alone
misses almost all of it.

### 6. The worst case, before you pay
You know the input exactly, and the output can be at most `max_output_tokens`. That gives an upper bound
for the turn, which you can check against a budget before calling.

In [9]:
def max_turn_cost(model: str, input_tokens: int, max_output_tokens: int) -> float:
    return cost_usd(model, input_tokens, max_output_tokens)

counted = full.total_tokens if "full" in globals() else 146   # 146 = recorded count if step 5 was skipped
worst = max_turn_cost(MODEL, counted, 60)
actual = cost_usd(MODEL, tokens_in, tokens_out)
print(f"worst case ${worst:.6f}  vs  actual ${actual:.6f}")

worst case $0.000194  vs  actual $0.000079


A 60-token cap keeps the worst case close to the real cost. With no cap the bound would be the model's
full output limit. Lesson 05 reserved room for the reply in its window check; here the same cap bounds the cost.

### 7. Different tokenizers, different counts
A quick revisit of lesson 03, which counted note 5 and three tricky texts (French, Japanese, digits) with
OpenAI's `o200k_base`. Its recorded output is in `../data/recorded/03.json`. Ask Gemini to count the same texts.

In [10]:
import json, pathlib
o200k = json.loads(pathlib.Path("../data/recorded/03.json").read_text())["o200k_base"]
TEXTS = [NOTES[4]["text"], "Le rover Perseverance a atterri sur Mars en février 2021.",
         "火星には二つの小さな衛星があります。", "3.14159265358979"]     # picked by name from lesson 03
for text in TEXTS:
    gemini_count = client.models.count_tokens(model=MODEL, contents=text).total_tokens
    print(f"o200k_base {len(o200k[text]):>2} | Gemini {gemini_count:>2} | {text[:40]}")

o200k_base 14 | Gemini 14 | The rover Perseverance landed on Mars in
o200k_base 17 | Gemini 17 | Le rover Perseverance a atterri sur Mars
o200k_base 13 | Gemini  9 | 火星には二つの小さな衛星があります。
o200k_base  7 | Gemini 16 | 3.14159265358979


English and French agree here. Japanese and the long number do not, and in opposite directions:
Gemini uses fewer tokens for Japanese and more for digits.
Bill a Gemini call with Gemini's count (or its `usage`), never with tiktoken.

### 8. Scout prints its bill after every turn
Lesson 08's `ask`, now with a running total. A small `Ledger` keeps each turn's tokens and cost.

In [11]:
class Ledger:
    def __init__(self, model: str):
        self.model, self.turns = model, []          # one (input, output, cost) tuple per turn

    def add(self, tokens_in: int, tokens_out: int) -> float:
        c = cost_usd(self.model, tokens_in, tokens_out)
        self.turns.append((tokens_in, tokens_out, c))
        return c

    @property
    def total(self) -> float:
        return sum(t[2] for t in self.turns)

def ask(messages: list[dict], user_text: str, ledger: Ledger) -> None:
    messages.append({"role": "user", "text": user_text})
    check_messages(messages)
    system, contents = to_gemini(messages)
    config = types.GenerateContentConfig(system_instruction=system, max_output_tokens=60,
                                         automatic_function_calling=NO_AFC)
    resp = client.models.generate_content(model=ledger.model, contents=contents, config=config)
    messages.append({"role": "assistant", "text": resp.text.strip()})
    c = ledger.add(*usage_tokens(resp))
    print(f"[{ledger.turns[-1][0]:>4} in] ${c:.5f} this turn, ${ledger.total:.4f} so far | {user_text[:38]}")

Ten turns of a normal research chat. Watch the input column.

In [12]:
QUESTIONS = [
    "Hi, I'm Asha. I'm researching Mars.", "Which notes mention Mars?", "How long is a day on Mars?",
    "What are the names of its moons?", "When did Perseverance land?", "Which planet is the largest?",
    "Is there anything about Venus?", "What are Saturn's rings made of?",
    "Which of my notes is not about a planet's physical features?", "What is my name and my topic?",
]
import json, pathlib
chat, ledger = [{"role": "system", "text": SYSTEM_TEXT}], Ledger(MODEL)
for q in QUESTIONS:
    ask(chat, q, ledger)
pathlib.Path("../data/recorded/12.json").write_text(json.dumps(ledger.turns))   # used by the solutions
print(f"total: ${ledger.total:.4f} for {len(ledger.turns)} turns")

[ 154 in] $0.00012 this turn, $0.0001 so far | Hi, I'm Asha. I'm researching Mars.
[ 190 in] $0.00009 this turn, $0.0002 so far | Which notes mention Mars?
[ 211 in] $0.00010 this turn, $0.0003 so far | How long is a day on Mars?
[ 235 in] $0.00010 this turn, $0.0004 so far | What are the names of its moons?
[ 252 in] $0.00011 this turn, $0.0005 so far | When did Perseverance land?
[ 272 in] $0.00011 this turn, $0.0006 so far | Which planet is the largest?
[ 288 in] $0.00011 this turn, $0.0007 so far | Is there anything about Venus?
[ 307 in] $0.00012 this turn, $0.0009 so far | What are Saturn's rings made of?
[ 332 in] $0.00014 this turn, $0.0010 so far | Which of my notes is not about a plane
[ 357 in] $0.00013 this turn, $0.0011 so far | What is my name and my topic?


total: $0.0011 for 10 turns


If you skipped the chat, load its recorded turns (saved by the cell above on a real run).

In [13]:
import json, pathlib
if "ledger" not in globals():   # recorded output: (input, output, cost) per turn
    ledger = Ledger(MODEL)
    ledger.turns = [tuple(t) for t in json.loads(pathlib.Path("../data/recorded/12.json").read_text())]
print(f"{len(ledger.turns)} turns, total ${ledger.total:.4f}")

10 turns, total $0.0011


Each turn's input is bigger than the last: the whole chat so far is sent again. The cost per turn rises
with it, and the total keeps climbing to $0.0011 for the 10 turns.

### 9. Why the total grows with the square of the length
Turn *i* sends the system prompt (size S) plus everything said before it, about *k* tokens per earlier
turn. So input on turn *i* is about S + k·(i−1), and the total over *n* turns is n·S + k·n(n−1)/2.
The second part has n², so it wins in long chats. We read S and k off the real chat.

In [14]:
inputs = [t[0] for t in ledger.turns]
avg_out = sum(t[1] for t in ledger.turns) / len(ledger.turns)
S = inputs[0]
k = (inputs[-1] - inputs[0]) / (len(inputs) - 1)       # average growth per turn
print(f"S = {S} tokens, k = {k:.1f} tokens per turn, average output {avg_out:.0f} tokens")

def chat_cost(n: int) -> float:
    total_in = n * S + k * n * (n - 1) / 2
    return cost_usd(MODEL, round(total_in), round(n * avg_out))

print(f"model for 10 turns: ${chat_cost(10):.4f}   real 10-turn chat: ${ledger.total:.4f}")

S = 154 tokens, k = 22.6 tokens per turn, average output 14 tokens
model for 10 turns: $0.0011   real 10-turn chat: $0.0011


The simple formula lands close to the real bill. Now project it to longer chats.

In [15]:
prev = None
for n in [10, 20, 40, 80, 160]:
    c = chat_cost(n)
    growth = f"x{c / prev:.1f}" if prev else ""
    print(f"{n:>4} turns: ${c:.4f} total, ${c / n:.5f} per turn on average {growth}")
    prev = c

  10 turns: $0.0011 total, $0.00011 per turn on average 
  20 turns: $0.0029 total, $0.00015 per turn on average x2.6
  40 turns: $0.0086 total, $0.00021 per turn on average x2.9
  80 turns: $0.0280 total, $0.00035 per turn on average x3.3
 160 turns: $0.0992 total, $0.00062 per turn on average x3.5


Each time the chat doubles, the total cost more than doubles, heading towards x4 as the n² part takes over.
The average turn gets dearer too. A support chat that runs for 160 turns is not 16 times the price of a
10-turn one; it is much more. The fixes (caching, trimming, compaction) come later, see At work.

## At work
- **Teams log tokens and cost on every call,** with model, request id and user, so they can answer "what did
  this feature cost yesterday?" A price table in config, not hard-coded constants. → lesson 49
- **Long chats get expensive fast,** exactly as step 9 shows. Production apps cap history, summarise old turns,
  or cache the repeated prefix (cache reads are billed at about 10% of the input price). → lessons 32 and 41
- **Budgets are enforced before the call,** using a token count and the `max_output_tokens` worst case, plus
  provider-side monthly spend limits set below the tier cap. → lesson 54
- **The system prompt and tool definitions are billed on every turn.** Anthropic adds about 286 hidden tokens
  for tool use on its large models, before your schemas; trimming them pays off on every call. → lessons 22 and 29

## What just happened
- `cost_usd` turned token counts into dollars with a per-million price table: 1,000 in + 200 out cost $0.0008.
- `usage_tokens` read input from `prompt_token_count` and output as candidates plus thinking, defaulting `None` to 0.
- `count_tokens` with the system instruction matched the real call's input count; the question alone was a tiny fraction.
- `max_turn_cost` gave an upper bound before sending, from the exact input and the output cap.
- Scout's 10-turn chat printed "$ this turn, $ so far"; input grew every turn because history is resent.
- A two-number model (S, k) matched the real total and showed cost roughly x4 per doubling of chat length.

## Common mistakes
**1. Adding usage fields that can be `None`.** With no thinking, `thoughts_token_count` is `None`, not 0.

In [16]:
u = resp.usage_metadata
output_tokens = u.candidates_token_count + u.thoughts_token_count

TypeError: unsupported operand type(s) for +: 'int' and 'NoneType'

Fix: default every optional count to 0, as `usage_tokens` does: `(u.thoughts_token_count or 0)`.

**2. Treating the price as per token.** The table says 0.30, and someone multiplies tokens by it directly.
The bill comes out a million times too high. A sanity check catches it:

In [17]:
tokens_in, tokens_out = usage_tokens(resp)
wrong = tokens_in * PRICES_PER_MTOK[MODEL]["input"] + tokens_out * PRICES_PER_MTOK[MODEL]["output"]
assert wrong < 0.01, f"one short turn costs ${wrong:.2f}? The prices are per MILLION tokens."

AssertionError: one short turn costs $78.80? The prices are per MILLION tokens.

Fix: divide by 1,000,000, in one place (`cost_usd`), and name the table so the unit is obvious
(`PRICES_PER_MTOK`).

## Try it
No new model calls needed: reuse `ledger.turns` from step 8.
1. Recompute the 10-turn chat's total cost as if it had run on `gemini-3.8-flash`. How many times dearer is it?
   (Its price doubles after 2026-12-31; rerun with 1.50 / 7.50.)
2. Using `chat_cost`, find the smallest number of turns at which a single chat on `MODEL` costs more than $0.05.

In [18]:
# your code here

Finally, stop the recorder (in record mode this saves the recording).

In [19]:
llm_replay.stop()

## Key terms
- **usage accounting** — reading the token counts each response reports (input, output, thinking) and turning them into cost per turn and per conversation.
- **price per million tokens (per MTok)** — how providers quote prices, separately for input and output; cost = tokens × price / 1,000,000.
- **count_tokens** — a provider call that returns how many input tokens a request would use, without generating; Gemini's is `client.models.count_tokens`.
- **worst-case turn cost** — exact input cost plus `max_output_tokens` at the output price; an upper bound you can check before sending.
- **cumulative chat cost** — the running total over a conversation; because history is resent, it grows roughly with the square of the number of turns.